# Contribution recovery · run-bound evidence

**Run All reads saved, hashed artifact inputs and renders only.** It cannot generate or fit. Legacy records are manifest/input-hash compatible only; historical smoke receipts are verified under their *recorded* identities, never recertified as current fits. All scientific conclusions remain **pending review for this exact run**.

In [1]:
from pathlib import Path
import sys
sys.path.insert(0, str((Path.cwd() / '../../../..').resolve()))
from studies.recovery.reporting import read_report
from IPython.display import display, Markdown
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
evidence = read_report(Path.cwd())
display(Markdown(f'**Evidence ID:** `{evidence["run_id"]}` · **Source:** `{evidence["kind"]}` · '
                 f'**Provenance:** {evidence["provenance"]}'))

**Evidence ID:** `041a2b4bfc17d53631ad524508fe0a6243c8bdf50df47ec3ee567a7c202562de` · **Source:** `bounded_ols` · **Provenance:** legacy manifest/input-hash compatibility only; output and source hashes unavailable

## Configuration, provenance and coverage

The evidence ID hashes the resolved config, input files, report sources and environment. Changed inputs fail Run All. Eligible channels exclude exact known-zero effects; a zero denominator means **not represented**, not zero failures. Small groups (<10 evaluable channels) are flagged.

In [2]:
display(Markdown('**Resolved configuration:**'))
display(pd.json_normalize(evidence['resolved_config']).T.rename(columns={0: 'resolved value'}))
display(pd.DataFrame([{'method': k, 'worlds': len({r['world_id'] for r in evidence['worlds'] if r.get('method', evidence['kind']) == k}), 'channels': sum(1 for c in evidence['channels'] if c.get('method', evidence['kind']) == k), 'eligible': sum(c['nonzero_effect'] for c in evidence['channels'] if c.get('method', evidence['kind']) == k)} for k in evidence['pattern_rates']]))
display(Markdown('**Optional exploration:** ' + str(evidence['capabilities'])))

**Resolved configuration:**

,resolved value
schema,1
diagnostics.bfmi,0.3
diagnostics.divergences,0
diagnostics.ess,400
diagnostics.maxdepth_hits,0
diagnostics.rhat,1.01
generator.carryover_family,geometric
generator.edge_budget.cc,"[0, 0]"
generator.edge_budget.cy,"[1, 15]"
generator.edge_budget.dc,"[0, 0]"


,method,worlds,channels,eligible
0,bounded_ols,50,799,384


**Optional exploration:** {'optional_logistic_screen': 'deferred/unsupported', 'optional_tsne': 'deferred/unsupported', 'optional_correlated_pairs': 'deferred/unsupported'}

## Noise, true parameters and variation

These are **oracle descriptive profiles**, not fit initialization and not predictors of recovery. Noise SD, baseline variation, media/control contribution SD, carryover alpha, MM multiplier and direct coefficients come from saved worlds.

In [3]:
profiles = pd.json_normalize([{'world': r['world_id'], 'method': r.get('method', evidence['kind']), **r['profile']} for r in evidence['worlds']])
display(profiles[['world', 'method', 'realized_noise_sd', 'baseline_temporal_sd', 'carryover_alpha', 'mm_kappa_mult', 'media_beta', 'control_coefficient', 'media_true_temporal_sd', 'control_true_temporal_sd']].head(60))

,world,method,realized_noise_sd,baseline_temporal_sd,carryover_alpha,mm_kappa_mult,media_beta,control_coefficient,media_true_temporal_sd,control_true_temporal_sd
0,0,bounded_ols,0.315691,0.0,"[0.2369566410779953, 0.5683885216712952, 0.518...","[11.978698219438716, 0.07262298481904934, 3.80...","[9.800710548456635, 1.9169488226177898, 9.2557...","[-0.11812368866036463, 0.03488309766867398, -0...","[0.0, 0.26858849615731695, 0.9209901691508734,...","[0.0, 0.0, 0.04275868584238596]"
1,1,bounded_ols,0.508518,0.0,"[0.5992437601089478, 0.5586657524108887, 0.548...","[0.13122343789213747, 19.522093630987555, 18.0...","[1.2602244107891194, 27.107757575937807, 24.21...","[0.11815139276876277, 0.14041838546046895, -0....","[0.16079334949113896, 0.2886103861680662, 0.16...","[0.01712255977963738, 0.027090324349444663, 0...."
2,2,bounded_ols,0.104234,0.0,"[0.7535253763198853, 0.2065604329109192, 0.475...","[2.7720702342362147, 0.06997015001664085, 4.87...","[4.320941180856281, 0.8301907628735165, 10.063...","[0.08797597296601845, 0.05125281931954234, -0....","[0.2337470445744714, 0.0, 0.0, 0.0717390315349...","[0.10741934675276345, 0.0, 0.0, 0.0, 0.0860611..."
3,3,bounded_ols,0.341557,0.0,"[0.36873242259025574, 0.5767415761947632, 0.76...","[10.49599799044622, 0.21310163093369586, 0.633...","[13.379044282151, 1.8779417580932, 1.326859943...","[0.010172471934946137, 0.06570099614669675, -0...","[0.3029368986249495, 0.11796947866432377, 0.10...","[0.0, 0.0, 0.0, 0.0, 0.021475551900879183, 0.0..."
4,4,bounded_ols,0.149083,0.0,"[0.7817994356155396, 0.6081915497779846, 0.560...","[12.129046556640644, 0.19956457988281284, 0.28...","[26.220762918799107, 2.22348773149627, 1.34523...","[-0.10773505067690592, 0.11062700294980607, -0...","[0.0, 0.0, 0.0, 0.0, 0.44343777398069445, 0.0,...","[0.0, 0.04471783854829004, 0.09672474798116945..."
5,5,bounded_ols,0.377748,0.0,"[0.2543151378631592, 0.25819700956344604, 0.71...","[0.5572417617404685, 0.5349406529763575, 0.160...","[1.6802851360243956, 1.7017982070956756, 1.956...","[0.06014740850041167, -0.009809137056882328, -...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.7414027581476...","[0.0, 0.0, 0.0, 0.0, 0.01862499905216813, 0.0,..."
6,6,bounded_ols,0.160157,0.0,"[0.2527131736278534, 0.29615518450737, 0.61198...","[0.11099265906353192, 0.7077022147963873, 4.57...","[0.6156601118974282, 1.9765241773540496, 4.924...","[0.14770123384788983, -0.13252530792165526, -0...","[0.0139217982242804, 0.49473053169555464, 0.42...","[0.2727870001452052, 0.24917411068031897, 0.06..."
7,7,bounded_ols,0.340967,0.0,"[0.7179519534111023, 0.2091435194015503, 0.486...","[2.040634096878339, 0.07900162235086992, 1.215...","[2.0353214334784977, 0.8520630460231245, 1.166...","[0.018293359453413593, 0.13976017483644845, 0....","[0.0, 0.053757230752331506, 0.2623099684640309...","[0.0, 0.0, 0.0, 0.0, 0.0]"
8,8,bounded_ols,0.247196,0.0,"[0.4183962047100067, 0.6244600415229797, 0.743...","[0.9321868626501912, 0.06538197676059651, 10.3...","[3.7614414265147955, 0.8683454461544444, 14.66...","[-0.03457852570527911, 0.11849542409193844, -0...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.43516778833424963,...","[0.0, 0.107140500710213, 0.1198429332322585, 0..."
9,9,bounded_ols,0.809067,0.0,"[0.32984814047813416, 0.2837252616882324, 0.58...","[1.839088581635389, 1.113206396367286, 0.19742...","[2.922496062970062, 3.996947056017796, 0.79562...","[0.09364271919003933, -0.0663289814390201, 0.0...","[0.12341152429889922, 0.8464139728055509, 0.0,...","[0.0, 0.0028081796163179256, 0.025831040323451..."


## Computational diagnostics ≠ point recovery ≠ uncertainty

Invalid MCMC exploration is unresolved, *not* a point failure. Short-smoke ADVI finite optimization only is **not** a convergence certificate; R-hat/ESS do not apply to variational draws. OLS has no posterior uncertainty. Pointwise credible intervals describe saved posterior uncertainty, not causal or frequentist generalization to new datasets.

In [4]:
diagnostics = pd.json_normalize([{'world': r['world_id'], 'method': r.get('method', evidence['kind']), 'status': r['computational_status'], **r['diagnostics']} for r in evidence['worlds']])
display(diagnostics.head(60))
recovery = pd.DataFrame([{'world': r['world_id'], 'method': r.get('method', evidence['kind']), 'status': r['computational_status'], 'media': r['recovery']['media_pass'], 'controls': r['recovery']['control_pass'], 'joint': r['recovery']['joint_pass'], 'total_media': (r['recovery']['total_media'] or {}).get('pass'), 'baseline': (r['recovery']['baseline'] or {}).get('pass'), 'uncertainty': r['uncertainty']['type']} for r in evidence['worlds']])
display(recovery.head(60))

,world,method,status,algorithm,optimizer_converged
0,0,bounded_ols,valid,bounded estimated-transform nonlinear least sq...,True
1,1,bounded_ols,valid,bounded estimated-transform nonlinear least sq...,True
2,2,bounded_ols,valid,bounded estimated-transform nonlinear least sq...,True
3,3,bounded_ols,valid,bounded estimated-transform nonlinear least sq...,True
4,4,bounded_ols,valid,bounded estimated-transform nonlinear least sq...,True
5,5,bounded_ols,valid,bounded estimated-transform nonlinear least sq...,True
6,6,bounded_ols,unresolved,bounded estimated-transform nonlinear least sq...,False
7,7,bounded_ols,valid,bounded estimated-transform nonlinear least sq...,True
8,8,bounded_ols,valid,bounded estimated-transform nonlinear least sq...,True
9,9,bounded_ols,unresolved,bounded estimated-transform nonlinear least sq...,False


,world,method,status,media,controls,joint,total_media,baseline,uncertainty
0,0,bounded_ols,valid,False,True,False,True,True,unavailable (point optimizer only)
1,1,bounded_ols,valid,False,False,False,False,False,unavailable (point optimizer only)
2,2,bounded_ols,valid,False,True,False,True,True,unavailable (point optimizer only)
3,3,bounded_ols,valid,False,True,False,False,False,unavailable (point optimizer only)
4,4,bounded_ols,valid,True,True,True,True,True,unavailable (point optimizer only)
5,5,bounded_ols,valid,False,False,False,True,False,unavailable (point optimizer only)
6,6,bounded_ols,unresolved,None,None,None,None,None,unavailable (point optimizer only)
7,7,bounded_ols,valid,False,True,False,False,True,unavailable (point optimizer only)
8,8,bounded_ols,valid,True,True,True,True,True,unavailable (point optimizer only)
9,9,bounded_ols,unresolved,None,None,None,None,None,unavailable (point optimizer only)


## Pattern-specific **channel** failure rates

Rows count **the channels bearing the named pattern**, not all failures in the same world. Patterns overlap and upward/downward trends have distinct denominators. Total includes known-zero channels; eligible excludes known-zero; unresolved eligible channels do not enter the failure-rate denominator.

In [5]:
for method, rates in evidence['pattern_rates'].items():
    display(Markdown('### ' + method))
    display(pd.DataFrame(rates)[['kind','pattern','total','eligible','known_zero','evaluable','failed','unresolved','failure_rate','status','small_count_warning']])
channel_table = pd.DataFrame(evidence['channels'])
display(channel_table.head(25))

### bounded_ols

,kind,pattern,total,eligible,known_zero,evaluable,failed,unresolved,failure_rate,status,small_count_warning
0,media,hf,441,214,227,193,109,21,0.564767,evaluable,False
1,media,pulse,340,156,184,139,79,17,0.568345,evaluable,False
2,media,onset,80,37,43,34,12,3,0.352941,evaluable,False
3,media,offset,39,17,22,16,5,1,0.312500,evaluable,False
4,media,flighting,93,45,48,39,16,6,0.410256,evaluable,False
5,media,level_jump,76,34,42,28,19,6,0.678571,evaluable,False
6,media,seasonal,137,73,64,71,32,2,0.450704,evaluable,False
7,media,trend,146,67,79,60,32,7,0.533333,evaluable,False
8,media,trend_up,72,35,37,32,17,3,0.531250,evaluable,False
9,media,trend_down,74,32,42,28,15,4,0.535714,evaluable,False


,world_id,kind,channel,patterns,nonzero_effect,computational_status,point_pass,point_error,point_allowance
0,0,media,C1,"[hf, pulse, offset, flighting, trend, trend_up]",False,valid,None,0.000000,0.886255
1,0,media,C2,"[hf, pulse, onset]",True,valid,True,0.983945,1.705871
2,0,media,C3,"[hf, pulse, onset, level_jump, trend, trend_up]",True,valid,False,3.216130,2.814847
3,0,media,C4,"[hf, onset, offset, seasonal]",False,valid,None,0.000000,0.886255
4,0,media,C5,"[hf, offset, level_jump, seasonal]",True,valid,True,1.051527,1.073390
5,0,control,Z1,"[hf, flighting, level_jump]",False,valid,None,0.000000,0.886255
6,0,control,Z2,"[hf, pulse, level_jump, seasonal]",False,valid,None,0.000000,0.886255
7,0,control,Z3,"[hf, offset, level_jump, seasonal]",True,valid,True,0.151838,0.886255
8,1,media,C1,"[hf, flighting, seasonal]",True,valid,False,8.407421,1.191112
9,1,media,C2,"[hf, trend, trend_down]",True,valid,False,2.092264,1.191112


## Selectable true/fitted trajectories

Select a world/method and a **nonzero** contribution. Separate Plotly panels show its contribution (and available 95% interval), raw media spend for media channels, and the baseline (always retained). Legends sit below the panels. Known-zero padded/null channels are hidden; absence is not a success.

In [6]:
keys = sorted(evidence['trajectories'])
selection = widgets.Dropdown(options=keys, value=keys[0], description='World · method:', layout=widgets.Layout(width='450px'))
def _eligible(key):
    if key not in evidence['trajectories']:
        return ('Baseline',)
    t = evidence['trajectories'][key]
    truth = np.asarray(t['truth'])
    return tuple(name for i, name in enumerate(t['terms']) if i == 0 or np.linalg.norm(truth[:, i]) > 0)
initial = _eligible(selection.value)
channel = widgets.Dropdown(options=initial, value=initial[0], description='Channel:', layout=widgets.Layout(width='300px'))
def _update_channels(change=None):
    valid = _eligible(selection.value)
    wanted = channel.value if channel.value in valid else valid[0]
    channel.options = valid
    channel.value = wanted
selection.observe(_update_channels, names='value')
def _view(key, name):
    if key not in evidence['trajectories'] or name not in _eligible(key):
        return None  # transient selector update, not a report/plot error
    t = evidence['trajectories'][key]
    ix = t['terms'].index(name)
    week = t['weeks']
    truth, fit = np.asarray(t['truth']), np.asarray(t['fitted'])
    fig = make_subplots(rows=3, cols=1, shared_xaxes=True, subplot_titles=(name + ' contribution', 'Raw media spend (if media)', 'Baseline contribution'))
    def line(row, values, label, dash='solid'):
        fig.add_trace(go.Scatter(x=week, y=values, name=label, mode='lines', line={'dash': dash}, legendgroup=label, showlegend=True), row=row, col=1)
    line(1, truth[:, ix], 'True ' + name)
    line(1, fit[:, ix], 'Fitted ' + name, 'dash')
    if t['lower95'] is not None:
        low, high = np.asarray(t['lower95'])[:, ix], np.asarray(t['upper95'])[:, ix]
        fig.add_trace(go.Scatter(x=week, y=low, line={'width':0}, showlegend=False, hoverinfo='skip'), row=1, col=1)
        fig.add_trace(go.Scatter(x=week, y=high, fill='tonexty', fillcolor='rgba(80,80,180,.17)', line={'width':0}, name='95% pointwise interval'), row=1, col=1)
    if name.startswith('C'):
        line(2, np.asarray(t['raw_spend'])[:, int(name[1:])-1], 'Raw spend ' + name)
    line(3, truth[:, 0], 'True baseline')
    line(3, fit[:, 0], 'Fitted baseline', 'dash')
    status = next((r['computational_status'] for r in evidence['worlds'] if str(r['world_id']) + (':' + r['method'] if 'method' in r else '') == key), 'unknown')
    fig.update_layout(height=800, template='plotly_white', legend={'orientation':'h', 'y':-.13, 'x':0}, title=key + ' · ' + name + ' · ' + status)
    display(fig)
    return fig
display(widgets.HBox([selection, channel]))
interactive = widgets.interactive_output(_view, {'key': selection, 'name': channel})
display(interactive)

Output()

## Interpretation · run-bound review

Pending until an explicit reviewer approves this exact run into a separate sealed sibling review artifact. A present but invalid/cross-run review FAILS CLOSED. Optional held-out-world logistic screening, t-SNE and correlated-pair analysis are **deferred/unsupported**.

In [7]:
from studies.recovery.review_interpretation import attached_review
review = attached_review(Path.cwd())
if review is None:
    display(Markdown(Path('interpretation.md').read_text()))
else:
    display(Markdown('**Reviewed for run** `' + review['report_run_id'] + '` by **' + review['reviewer'] + '** · decision: approved · review `' + review['review_id'] + '`'))
    display(Markdown(review['interpretation']))

**Reviewed for run** `041a2b4bfc17d53631ad524508fe0a6243c8bdf50df47ec3ee567a7c202562de` by **Teemu Säilynoja** · decision: approved · review `031ec6a9b96dfa8e7fdd2821cc25eabcef482c87a36082c2d62e81dc30f7939b`

# Interpretation: bounded least-squares contribution recovery

**Report:** `studies/recovery/report-artifacts/bounded50-profile-v2/`

**Exact report run ID:** `041a2b4bfc17d53631ad524508fe0a6243c8bdf50df47ec3ee567a7c202562de`

This narrative concerns the existing 50-world constant-baseline study. Approval, if granted, applies only to this report identity; it is recorded separately from this text.

## Main finding

**Optimizer convergence was much more common than successful channel attribution.** Among 48 computationally evaluable worlds, only 13 recovered all individual media contributions, whereas 28 recovered the total media contribution. This distinguishes numerical completion from scientific recovery and motivates keeping both checks in the study runner.

Here “OLS” names the report's method slot; the procedure is **bounded estimated-transform nonlinear least squares**, not ordinary unconstrained linear regression. It estimates adstock and saturation transformations under generator-supported constraints and uses known zero/nonzero effect identities. It returns point estimates, not a posterior distribution.

## Results and denominators

The study contains 50 worlds with constant intrinsic baselines and 97 scored weeks (7–103). Two optimizer runs, zero-based world IDs 6 and 9, did not satisfy the computational acceptance rule and remain unresolved. They are not counted as attribution failures or dropped without disclosure.

The point-recovery allowance is the greater of 10% of the true channel's L2 norm and 1% of the true noiseless outcome's L2 norm. Joint recovery covers nonzero media/control effects but excludes the baseline; total media and baseline are evaluated separately.

| Estimand | Passes / evaluable worlds | Rate |
|---|---:|---:|
| Every individual media contribution | 13/48 | 27.1% |
| Every individual media and control contribution | 13/48 | 27.1% |
| Total media contribution trajectory | 28/48 | 58.3% |
| Baseline contribution | 26/48 | 54.2% |

Relative to all 50 attempted worlds, those are 13, 13, 28 and 26 established passes, respectively, with two unresolved worlds. The table conditions on computational validity; it is not an unconditional performance estimate for all generated worlds.

| Nonzero effects | Failed / evaluable | Failure rate | Unresolved |
|---|---:|---:|---:|
| Media | 122/211 | 57.8% | 25 |
| Controls | 15/135 | 11.1% | 13 |

There are 236 eligible media and 148 eligible control effects overall. Known-zero effects are not part of these denominators. Channels within worlds are dependent, and worlds with more channels contribute more observations to pooled rates.

## Interpretation

The optimizer often finds a solution satisfying its stopping rule without finding contribution trajectories close to truth. The result is compatible with weak separation of component effects and with limitations of a finite optimization procedure; **convergence is not a certificate of a global optimum**. These results alone cannot distinguish structural ambiguity, weak finite-sample information, objective geometry and remaining optimization error.

Total-media point recovery is more frequent than recovery of every component, illustrating why an aggregate contribution estimate should not be advertised as channel-level attribution. Conversely, a failed all-channel gate need not mean that every channel or the aggregate estimate failed.

There are no posterior intervals in this point-estimation report, so it cannot establish uncertainty calibration, posterior concentration or credible-interval coverage. It also does not evaluate held-out predictions or interventions. The generator's known graph and supported transformations make this an oracle-information benchmark rather than a realistic variable-selection task.

The TV-MCMC report is useful complementary evidence but **not a controlled algorithm comparison**: it uses a changed baseline/outcome regime and a different estimation objective. These two headline rates do not support saying that one inference method is better than the other. A method comparison would require the same generated worlds, target, information and scoring conditions, with comparable computational qualification.

## Implications and provenance

Keep optimizer acceptance, channel recovery and aggregate recovery separate. Preserve unresolved denominators when summarizing future studies. Test generator or fitting changes through prespecified matched comparisons rather than interpreting these exploratory failure counts as a causal explanation.

This legacy evidence has manifest/input-hash compatibility only; unavailable original per-output/source hashes cannot be certified retrospectively. The report seals its current evidence without upgrading that historical provenance. The four-world software smoke is not additional scientific evidence, and no new fitting was performed to prepare this interpretation.
